# 第5段階　段位期と打牌特徴量の関係は将来期間にも続くか
しょぼんｎのFold 2・3の学習窓と評価窓を比較します。主分析は非重複10対局窓、感度分析は20対局窓です。第3段階の保存予測と第4段階の結果を照合し、ウルトラ立直との比較には既存20特徴量を使用します。R1〜R4を順に実行してください。

In [ ]:
# R1　Drive接続・分析コード取得・合成データでの検査
from google.colab import drive
from pathlib import Path
import sys, os, subprocess, tempfile, urllib.request, importlib.util
drive.mount('/content/drive')
PIN = '93e1fe140b19987dfcfd06448ba09141f01cd1da'
src = Path(tempfile.mkdtemp(prefix='mahjong_model_'))
files = ['mahjong_research/'+f for f in ('__init__.py','analysis.py','inputs.py','tenhou.py','legacy.py')]
files += ['smoothing/study.py','smoothing/presentation.py','smoothing/test_smoothing.py',
          'stability/stability.py','choices/choices.py']
files += ['model_comparison/'+f for f in ('windows.py','compare.py','outputs.py','test_comparison.py')]
files += ['error_analysis/'+f for f in ('investigate.py','diagnostics.py','findings.py','test_error_analysis.py')]
files += ['relationship_analysis/'+f for f in ('relationships.py','presentation5.py','test_relationships.py')]
for rel in files:
    target=src/rel
    target.parent.mkdir(exist_ok=True)
    url=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{PIN}/research_v2/{rel}'
    with urllib.request.urlopen(url,timeout=60) as r: target.write_bytes(r.read())
if importlib.util.find_spec('lightgbm') is None:
    subprocess.run([sys.executable,'-m','pip','install','lightgbm==4.6.0'],check=True)
paths=[str(src/p) for p in ('relationship_analysis','error_analysis','model_comparison','stability','smoothing','choices','.')]
env=dict(os.environ,PYTHONPATH=os.pathsep.join(paths),MPLBACKEND='Agg')
subprocess.run([sys.executable,'-m','unittest','-q','test_relationships'],cwd=src,env=env,check=True)

In [ ]:
# R2　Driveの入力フォルダを指定。第2段階は第3段階の入力ハッシュで照合
import hashlib, json
ROOT=Path('/content/drive/MyDrive/卒論')
RUN=ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'
PHASE1=ROOT/'research_v2_stability/20261105T092459Z_113b116b'
PHASE3=ROOT/'research_v2_model_comparison/20261117T160007Z_113bc0ca'
PHASE2=None  # 手動指定時はQ5完了フォルダのフルパス
ULTRA_CACHE=None  # C8キャッシュを移動した場合のみ指定
PHASE4=None  # 候補が複数ある場合は今回使用する第4段階の日時付きフォルダを指定
OUTPUT=ROOT/'research_v2_relationship_analysis'
if PHASE2 is None:
    signature=json.loads((PHASE3/'protocol.json').read_text())['inputs']['window_values.csv']
    matches=[p.parent for p in sorted((ROOT/'research_v2_choices').glob('*/window_values.csv'))
             if hashlib.sha256(p.read_bytes()).hexdigest()==signature]
    if not matches: raise FileNotFoundError('第3段階で使用したQ5出力をPHASE2に指定してください')
    PHASE2=matches[0]
if PHASE4 is None:
    protocol_hash=hashlib.sha256((PHASE3/'protocol.json').read_bytes()).hexdigest()
    candidates=[p.parent for p in sorted((ROOT/'research_v2_error_analysis').glob('*/sources.json'))
                if (p.parent/'report_ja.md').is_file() and json.loads(p.read_text(encoding='utf8')).get('protocol_sha256')==protocol_hash]
    if len(candidates)!=1:
        raise ValueError(f'対応する第4段階の完了フォルダをPHASE4に指定してください。候補 {candidates}')
    PHASE4=candidates[0]
print('第2段階',PHASE2,'第4段階',PHASE4,'出力の親',OUTPUT)

In [ ]:
# R3　保存された窓と予測を照合し、群別統計・時系列・外部比較・図表を保存
args=[str(p) for p in (RUN,PHASE1,PHASE2,PHASE3,PHASE4,OUTPUT)]+[str(ULTRA_CACHE) if ULTRA_CACHE else None]
code="""import sys,json
from pathlib import Path
from relationships import run
out=run(*json.loads(sys.argv[1]))
Path(sys.argv[2]).write_text(str(out),encoding='utf8')
"""
result_path=src/'result_path.txt'
subprocess.run([sys.executable,'-u','-c',code,json.dumps(args),str(result_path)],cwd=src,env=env,check=True)
RESULT=Path(result_path.read_text(encoding='utf8'))
print('保存先',RESULT)

In [ ]:
# R4　日本語レポート、実質的変化の大きい項目と重点5項目の図を表示
import pandas as pd
from IPython.display import display, Markdown, Image
display(Markdown((RESULT/'report_ja.md').read_text(encoding='utf8')))
effects=pd.read_csv(RESULT/'effect_size_comparison.csv')
display(effects.query('width==10 and fold==3').sort_values('change_rank'))
for name in ('focus_distributions_10.png','temporal_trends_10.png'):
    display(Image(filename=str(RESULT/name)))